# 3.0 Deep Learning Model Training & Transfer Learning

**Project:** Healthcare & Medical – Diagnostic Lab Image Classification  
**Organization:** Axlero  
**Milestone:** Phase 2 (Day 9) – Initial Model Training & Convergence Evaluation  
**Branch:** `Syed-Abdus-Sabur`  

---

## 1. Objectives & Training Setup
1. Load chest radiograph batches via stratified `create_dataloaders()`.
2. Apply class-weighted Cross-Entropy Loss to penalize false negatives (prioritizing recall > 90%).
3. Instantiate pretrained CNN transfer learning architectures (`ResNet50` and `DenseNet121`).
4. Train models using AdamW with weight decay regularization.
5. Track loss, accuracy, sensitivity (recall), and F1-score across epochs.
6. Persist model checkpoints to `models/`.


In [ ]:
import os
import sys
from pathlib import Path
import torch
import torch.nn as nn
from torch.optim import AdamW

project_root = Path('..').resolve() if Path('..').resolve().name != 'notebooks' else Path('.').resolve()
sys.path.insert(0, str(project_root))

from src.dataset import create_dataloaders
from src.model import build_model
from src.utils import get_device, set_seed, count_parameters

set_seed(42)
device = get_device()
print(f"Active Compute Device: {device}")


## 2. DataLoader & Class Weights Inspection
Inspecting stratified train, validation, and test splits with inverse-frequency class weighting.

In [ ]:
data_dir = project_root / 'data' / 'raw' / 'chest_xray'
loaders = create_dataloaders(data_dir=data_dir, batch_size=32, random_state=42)

train_loader = loaders['train']
val_loader = loaders['val']
class_weights = loaders['class_weights'].to(device)

print(f"Training Batches:   {len(train_loader)} ({len(loaders['train_dataset']):,} images)")
print(f"Validation Batches: {len(val_loader)} ({len(loaders['val_dataset']):,} images)")
print(f"Class Weights:      NORMAL={class_weights[0]:.3f}, PNEUMONIA={class_weights[1]:.3f}")


## 3. ResNet50 Transfer Learning Architecture
Initializing ResNet50 backbone with custom 2-class regularized classification head.

In [ ]:
resnet_model = build_model('resnet50', num_classes=2, pretrained=True, freeze_backbone=True)
resnet_model = resnet_model.to(device)

total_p, trainable_p = count_parameters(resnet_model)
print(f"ResNet50 Total Parameters:     {total_p:,}")
print(f"ResNet50 Trainable Parameters: {trainable_p:,} (Classification Head)")


## 4. DenseNet121 Transfer Learning Architecture
Initializing DenseNet121 backbone known for parameter efficiency and dense feature propagation.

In [ ]:
densenet_model = build_model('densenet121', num_classes=2, pretrained=True, freeze_backbone=True)
densenet_model = densenet_model.to(device)

total_d, trainable_d = count_parameters(densenet_model)
print(f"DenseNet121 Total Parameters:     {total_d:,}")
print(f"DenseNet121 Trainable Parameters: {trainable_d:,} (Classification Head)")


## 5. Loss Function & Optimization Strategy
- **Weighted CrossEntropyLoss:** Compensates for class imbalance.
- **AdamW:** Decouples weight decay (`1e-3`) from gradient updates for superior generalization.

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = AdamW(filter(lambda p: p.requires_grad, resnet_model.parameters()), lr=5e-4, weight_decay=1e-3)
print("Loss and Optimizer Configured Successfully.")


## 6. Training Execution & Checkpointing
Execute initial training session via `src/run_initial_training.py` and inspect learning curves in `reports/figures/training_curves.png`.